In [ ]:
#Autor: Felix Kirchhoff
#Matrikelnummer:  

In [ ]:
# Install Modules
# !pip install transformers
#!pip install torch
!pip install emoji
!pip install bertopic
# https://maartengr.github.io/BERTopic/index.html#installation
# install detoxify
!pip install detoxify
!pip install sklearn

  Using cached emoji-2.16.0-py3-none-any.whl.metadata (5.7 kB)
Using cached emoji-2.16.0-py3-none-any.whl (610 kB)


In [103]:
import pandas as pd
import regex as re
import numpy
import emoji
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from detoxify import Detoxify
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
import umap
from bertopic.representation import MaximalMarginalRelevance
from sklearn.cluster import KMeans

def create_tox_scores(data_frame_topics):
    # liste für tox scores anlegen
    toxicity_scores = []
    # Die Twitter Kommentare in einer liste abspeichern 
    clean_text_list = data_frame_topics["Clean_text"].tolist()

    start = 0
    # In 150er schritten durch die liste gehen 
    for i in range(0,len(clean_text_list),150):
        end = start + 150 
        if end <= len(clean_text_list):
            # hier wird eine Liste mit 150 Kommentaren auf einmal mittels Detoxify klassifiziert. Ich habe mich für 150 entschieden, weil es so zeiteffizient gelaufen ist. 
            result = Detoxify('unbiased').predict(clean_text_list[start:end])
        # overflow verhindern
        elif end > len(clean_text_list): 
            result = Detoxify('unbiased').predict(clean_text_list[start:len(clean_text_list)])
        toxicity = result["toxicity"]
        toxicity_scores.extend(toxicity)
        start = end
    #print(toxicity_scores)
    toxicity_scores_series = pd.Series(toxicity_scores, index=len_df)
    #print(len(toxicity_scores_series))
    # Dem Data Frame zuweisen 
    data_frame_topics["Toxicity_score"] = toxicity_scores_series

    return data_frame_topics

def improve_data_frame(data_set_no_spam):
    # Nur relevante Spalten auswählen und outlier entfernen
    # https://stackoverflow.com/a/31097813


    list_of_series = [data_set_no_spam.loc[data_set_no_spam["Topic"] != -1,"Clean_text"], data_set_no_spam.loc[data_set_no_spam["Topic"] != -1,"Topic"], data_set_no_spam.loc[data_set_no_spam["Topic"] != -1,"Topic_name"]]
    data_frame_topics = pd.concat(list_of_series, axis=1)
    # liste mit ids des Dataframes ersetllen
    len_df = range(len(data_frame_topics[data_frame_topics["Topic"] != -1]))
    data_frame_topics = data_frame_topics.set_index(pd.Index(len_df))

    return data_frame_topics


def map_topic_names(data_set_no_spam, topic_model):
    # Topic Names den IDs zuweisen 
    custom_labels = topic_model.custom_labels_ 
    #print(custom_labels)
    lable_dict = {}
    lable_dict[-1] = custom_labels[0]
    for i, lable in enumerate(custom_labels[1:]):
        lable_dict[i] = lable

    # Die Ids und die Topic Namen im Datensatz Mappen 
    data_set_no_spam["Topic_name"] = data_set_no_spam["Topic"].map(lable_dict)

    return data_set_no_spam


def merge_topics(topic_model):
    """
    -1 unfollowed, unfollowed automatically, followed, nigga, annoying, racist, niggas, know fucking, stop fucking, lil
    0 trump, realdonaldtrump, trumps, presidency, fox news, donald trump, obama, president, putin, white house - 
    1 giveaway, entered giveaway, tickets, enter, gift, ticket, entered, win, chance win, card
    2 pussy fucked, girls fucked, fucking sex, girl fucked, anal fucking, pussy, sex, pussy fucking, gets fucked, babe fucked
    3 attack syria, strikes syria, strike syria, bombing syria, war syria, syria, syria fucking, syrian, syriastrikes, missile strike
    4 verse, proverb, prayer, happiness, faith, praise, succeed, peace, success, spiritual
    5 bjp4india, bjp, delhi, crore, india, hindu, hindi, indian, shekhargupta, bhai
    6 newmusic, new music, songs, tunes, music, new song, remix, soundcloud, soundtrack, soundcloud np
    7 mlb wire, mlb, sox, baseball, astros, braves, cubs, inning, orioles, phillies
    8 episode, watched episode, season episode, prison break, episodes, 13 reasons, watch 13, finale, prisonbreak, game thrones
    9 ivy league, educational, scholarship, schools, education, courses, students, educators, college, conference
    10 naps risky, taking naps, naps, risky wake, wake, wake refreshed, lost feeling, getting sick, risky, like shit
    11 trust issues, trust, generation fucked, don trust, loyalty, generation, relationships, relationship, parents, people fucking
    12 manchester united, premier league, arsenal, tottenham, liverpool, manchester city, united, sunderland, chelsea, watford
    13 sweetie sorry, sorry ugly, sweetie, jasmine sweetie, sweetie im, ugly ass, ass bitch, im sorry, bitch say, bitch like
    14 protesting laws, know protesting, protesting, phone case, laws shoot, laws, case, stupid phone, damn phone, people mad
    15 birthday happy, red_heart happy, day red_heart, birthday love, happy bday, love red_heart, happy birthday, red_heart red_heart, red_heart love, birthday
    16 chicken nuggets, chicken, cook, cooking, recipe, meals, recipes, meal, food, dinner
    17 bitch bad, bad bitch, bad bitches, hate bitch, bitch bitch, bitch hate, hate bitches, bitches hate, bitch like, stupid bitch
    18 tweet fucking, fucking tweet, tweet, read tweets, fucking twitter, twitter fucking, tweets just, tweets, retweet, tweeting
    19 need sleep, sleeping schedule, sleep schedule, sleep, hours sleep, fuck sleep, fucking sleep, sleeping, falling asleep, slept
    20 video preorder, preorder, preorder lets, happening song, fucking nowornever, nowornever, song video, song, lets fucking, fucking disaster


    anhand der kategorien merge ich noch manuell kategorien zusammen
    """
    topics_to_merge = [[2,13,17], [10,19], [6,20]]

    topic_model.merge_topics(docs, topics_to_merge)

    # Die nun neuen Kategrien mit Custom Labels versehen 
    custom_labels = {-1: "Outlier", 0:"US-Politik & Trump",1:"Sexuelle Inhalte & Frauenfeindliche Beleidigungen",2:"Gewinnspiele & Verlosungen",3:"Musik und Videos",4: "Schlafen", 5:"Syrien-Krieg & Militärkonflikt", 6: "Religion & Spiritualität", 7: "Indische Politik & Indien", 8:"Baseball", 9:"TV-Serien", 10: "Bildung & Universität", 11: "Beziehungen & Vertrauen", 12: "Fußball", 13: "Proteste", 14: "Geburtstagsgrüße", 15: "Essen & Kochen", 16: "(Twitter) Following updates"}
    topic_model.set_topic_labels(custom_labels)

    # neue Topcis extrahieren, da sie sich nach dem mergen verändert haebn 
    new_topics = topic_model.topics_

    return new_topics

def create_topics(data_set_no_spam):
    """
    BERTopic Modell aufbereiten und anpassen. Dafür wurden folgende Quellen benutzt: 
    Die offizielle Doku habe ich komplett benutzt (auch die anderen Kapitel)
    https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html

    Weitere Quellen:
    https://towardsdatascience.com/a-practical-guide-to-bertopic-for-transformer-based-topic-modeling/
    https://towardsdatascience.com/finetune-your-topic-modeling-workflow-with-bertopic/
    """
    # Aufbereitete Daten an Liste übergeben 
    docs = data_set_no_spam["Clean_text"].tolist()
    # Create your representation model
    representation_model = [KeyBERTInspired(),MaximalMarginalRelevance(diversity=0.3)]

    # Erzeugt 384-Dimensionale Embeddings
    sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = sentence_model.encode(docs, show_progress_bar=True)

    umap_model = umap.UMAP(n_neighbors=10,min_dist=0.1,random_state=0)


    # Use the representation model in BERTopic on top of the default pipeline
    topic_model = BERTopic(representation_model=representation_model)


    hdbscan_model = HDBSCAN(min_cluster_size=800, metric='euclidean', cluster_selection_method='leaf', min_samples=50) # 150 # 20

    #clustering_model = KMeans(n_clusters=20,random_state=0)

    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words = True)

    # Use the representation model in BERTopic on top of the default pipeline
    topic_model = BERTopic(embedding_model=sentence_model,representation_model=representation_model, hdbscan_model=hdbscan_model, ctfidf_model=ctfidf_model, umap_model=umap_model) # calculate probs true -> für outlier reduction 
    topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

    # update topics
    vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1,2), min_df=2, max_df=0.8) # wörter eines topics müssen in 5% der texte vorkommen 
    topic_model.update_topics(docs, topics= topics ,vectorizer_model=vectorizer_model, representation_model=representation_model)

    new_topics = merge_topics(topic_model)
    
    # Topics dem datensatz zuweisen 
    data_set_no_spam["Topic"] = new_topics

    return data_set_no_spam, topic_model

def data_cleaning(text):
    # Datenbereinigung
    # Entfernen von: @, hastags, urls, emojis, HTML Codes 

    # Regex für ReTweets mit Namen 
    repost_reg = r"\bRT\s?@\w+:?" 
    # Reges für htm für die emojis und andere Codes 
    html_codes = r'&#?\w+;'
    # Regex für Links  -> entweder http oder https oder www
    urls = r'https?://\S+|www\.\S+'
    
    text = re.sub(repost_reg, "", text)
    text = re.sub(r"@", "", text)
    text = re.sub(html_codes, "", text)
    text = re.sub(urls, "", text)
    text = re.sub(r"#", "", text)
    # Emojis zu Text verarbeiten 
    text = emoji.demojize(text)
    # White spaces korrigieren 
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    # falls leere texte entstanden sind als NAN zurückgeben
    if text == "": 
        return numpy.nan
    return text 

def read_and_select_data_set(file_path):
    # Datensatz einlesen 
    data_set = pd.read_csv(file_path, delimiter="\t")
    # Spam aussortieren 
    data_set_no_spam = data_set[data_set["Label"] != "spam"]
    # Datensatz aufbereiten
    data_set_no_spam["Clean_text"] = data_set_no_spam["Tweet_text"].apply(data_cleaning)

    return data_set_no_spam

def run_script(file_path):
    
    data_set_no_spam = read_and_select_data_set(file_path)
    data_set_no_spam, topic_model = create_Topics(data_set_no_spam)
    data_set_no_spam = map_topic_names(data_set_no_spam,topic_model)
    data_frame_topics = improve_data_frame(data_set_no_spam)
    data_frame_topics = create_tox_scores(data_frame_topics)

    # Als csv Datei ausgeben
    data_frame_topics.to_csv("MAP_twitter_toxicity_topics.csv")
    
if __name__ == "__main__":

    file_path = "MAP_CL\data\hatespeech_text_label_vote_RESTRICTED_100K.csv"

    # Funktion, die alle weiteren Funktionen aufruft
    run_script(file_path)


In [ ]:
def data_cleaning(text):
    # Datenbereinigung
    # Entfernen von: @, hastags, urls, emojis, HTML Codes 

    # Regex für ReTweets mit Namen 
    repost_reg = r"\bRT\s?@\w+:?" 
    # Reges für htm für die emojis und andere Codes 
    html_codes = r'&#?\w+;'
    # Regex für Links  -> entweder http oder https oder www
    urls = r'https?://\S+|www\.\S+'
    
    text = re.sub(repost_reg, "", text)
    text = re.sub(r"@", "", text)
    text = re.sub(html_codes, "", text)
    text = re.sub(urls, "", text)
    text = re.sub(r"#", "", text)
    # Emojis zu Text verarbeiten 
    text = emoji.demojize(text)
    # White spaces korrigieren 
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    # falls leere texte entstanden sind als NAN zurückgeben
    if text == "": 
        return numpy.nan
    return text 

In [105]:
# Datensatz einlesen 
data_set = pd.read_csv("Praxis/MAP/hatespeech_text_label_vote_RESTRICTED_100K.csv", delimiter="\t")
# Spam aussortieren 
data_set_no_spam = data_set[data_set["Label"] != "spam"]
# Datensatz aufbereiten
data_set_no_spam["Clean_text"] = data_set_no_spam["Tweet_text"].apply(data_cleaning)

In [ ]:
"""
BERTopic Modell aufbereiten und anpassen. Dafür wurden folgende Quellen benutzt: 
Die offizielle Doku habe ich komplett benutzt (auch die anderen Kapitel)
https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html

Weitere Quellen:
https://towardsdatascience.com/a-practical-guide-to-bertopic-for-transformer-based-topic-modeling/
https://towardsdatascience.com/finetune-your-topic-modeling-workflow-with-bertopic/
"""
# Aufbereitete Daten an Liste übergeben 
docs = data_set_no_spam["Clean_text"].tolist()
# Create your representation model
representation_model = [KeyBERTInspired(),MaximalMarginalRelevance(diversity=0.3)]

# Erzeugt 384-Dimensionale Embeddings
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=True)

umap_model = umap.UMAP(n_neighbors=10,min_dist=0.1,random_state=0)


# Use the representation model in BERTopic on top of the default pipeline
topic_model = BERTopic(representation_model=representation_model)


hdbscan_model = HDBSCAN(min_cluster_size=800, metric='euclidean', cluster_selection_method='leaf', min_samples=50) # 150 # 20

#clustering_model = KMeans(n_clusters=20,random_state=0)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words = True)

# Use the representation model in BERTopic on top of the default pipeline
topic_model = BERTopic(embedding_model=sentence_model,representation_model=representation_model, hdbscan_model=hdbscan_model, ctfidf_model=ctfidf_model, umap_model=umap_model) # calculate probs true -> für outlier reduction 
topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

# update topics
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1,2), min_df=2, max_df=0.8) # wörter eines topics müssen in 5% der texte vorkommen 
topic_model.update_topics(docs, topics= topics ,vectorizer_model=vectorizer_model, representation_model=representation_model)

# -> update the topics and probs list? für das dataframe?
data_set_no_spam["Topic"] = topics

In [ ]:
"""
-1 unfollowed, unfollowed automatically, followed, nigga, annoying, racist, niggas, know fucking, stop fucking, lil
0 trump, realdonaldtrump, trumps, presidency, fox news, donald trump, obama, president, putin, white house - 
1 giveaway, entered giveaway, tickets, enter, gift, ticket, entered, win, chance win, card
2 pussy fucked, girls fucked, fucking sex, girl fucked, anal fucking, pussy, sex, pussy fucking, gets fucked, babe fucked
3 attack syria, strikes syria, strike syria, bombing syria, war syria, syria, syria fucking, syrian, syriastrikes, missile strike
4 verse, proverb, prayer, happiness, faith, praise, succeed, peace, success, spiritual
5 bjp4india, bjp, delhi, crore, india, hindu, hindi, indian, shekhargupta, bhai
6 newmusic, new music, songs, tunes, music, new song, remix, soundcloud, soundtrack, soundcloud np
7 mlb wire, mlb, sox, baseball, astros, braves, cubs, inning, orioles, phillies
8 episode, watched episode, season episode, prison break, episodes, 13 reasons, watch 13, finale, prisonbreak, game thrones
9 ivy league, educational, scholarship, schools, education, courses, students, educators, college, conference
10 naps risky, taking naps, naps, risky wake, wake, wake refreshed, lost feeling, getting sick, risky, like shit
11 trust issues, trust, generation fucked, don trust, loyalty, generation, relationships, relationship, parents, people fucking
12 manchester united, premier league, arsenal, tottenham, liverpool, manchester city, united, sunderland, chelsea, watford
13 sweetie sorry, sorry ugly, sweetie, jasmine sweetie, sweetie im, ugly ass, ass bitch, im sorry, bitch say, bitch like
14 protesting laws, know protesting, protesting, phone case, laws shoot, laws, case, stupid phone, damn phone, people mad
15 birthday happy, red_heart happy, day red_heart, birthday love, happy bday, love red_heart, happy birthday, red_heart red_heart, red_heart love, birthday
16 chicken nuggets, chicken, cook, cooking, recipe, meals, recipes, meal, food, dinner
17 bitch bad, bad bitch, bad bitches, hate bitch, bitch bitch, bitch hate, hate bitches, bitches hate, bitch like, stupid bitch
18 tweet fucking, fucking tweet, tweet, read tweets, fucking twitter, twitter fucking, tweets just, tweets, retweet, tweeting
19 need sleep, sleeping schedule, sleep schedule, sleep, hours sleep, fuck sleep, fucking sleep, sleeping, falling asleep, slept
20 video preorder, preorder, preorder lets, happening song, fucking nowornever, nowornever, song video, song, lets fucking, fucking disaster


anhand der kategorien merge ich noch manuell kategorien zusammen
"""
topics_to_merge = [[2,13,17], [10,19], [6,20]]

topic_model.merge_topics(docs, topics_to_merge)

# Die nun neuen Kategrien mit Custom Labels versehen 
custom_labels = {-1: "Outlier", 0:"US-Politik & Trump",1:"Sexuelle Inhalte & Frauenfeindliche Beleidigungen",2:"Gewinnspiele & Verlosungen",3:"Musik und Videos",4: "Schlafen", 5:"Syrien-Krieg & Militärkonflikt", 6: "Religion & Spiritualität", 7: "Indische Politik & Indien", 8:"Baseball", 9:"TV-Serien", 10: "Bildung & Universität", 11: "Beziehungen & Vertrauen", 12: "Fußball", 13: "Proteste", 14: "Geburtstagsgrüße", 15: "Essen & Kochen", 16: "(Twitter) Following updates"}
topic_model.set_topic_labels(custom_labels)

# neue Topcis extrahieren, da sie sich nach dem mergen verändert haebn 
new_topics = topic_model.topics_

# Topics dem datensatz zuweisen 
data_set_no_spam["Topic"] = new_topics


In [ ]:
# Topic Names den IDs zuweisen 
custom_labels = topic_model.custom_labels_ 
#print(custom_labels)
lable_dict = {}
lable_dict[-1] = custom_labels[0]
for i, lable in enumerate(custom_labels[1:]):
    lable_dict[i] = lable

# Die Ids und die Topic Namen im Datensatz Mappen 
data_set_no_spam["Topic_name"] = data_set_no_spam["Topic"].map(lable_dict)

In [139]:
# Nur relevante Spalten auswählen und outlier entfernen
# https://stackoverflow.com/a/31097813


list_of_series = [data_set_no_spam.loc[data_set_no_spam["Topic"] != -1,"Clean_text"], data_set_no_spam.loc[data_set_no_spam["Topic"] != -1,"Topic"], data_set_no_spam.loc[data_set_no_spam["Topic"] != -1,"Topic_name"]]
data_frame_topics = pd.concat(list_of_series, axis=1)
# liste mit ids des Dataframes ersetllen
len_df = range(len(data_frame_topics[data_frame_topics["Topic"] != -1]))
data_frame_topics = data_frame_topics.set_index(pd.Index(len_df))



In [ ]:
# liste für tox scores anlegen
toxicity_scores = []
# Die Twitter Kommentare in einer liste abspeichern 
clean_text_list = data_frame_topics["Clean_text"].tolist()

start = 0
# In 150er schritten durch die liste gehen 
for i in range(0,len(clean_text_list),150):
    end = start + 150 
    if end <= len(clean_text_list):
        # hier wird eine Liste mit 150 Kommentaren auf einmal mittels Detoxify klassifiziert. Ich habe mich für 150 entschieden, weil es so zeiteffizient gelaufen ist. 
        result = Detoxify('unbiased').predict(clean_text_list[start:end])
    # overflow verhindern
    elif end > len(clean_text_list): 
        result = Detoxify('unbiased').predict(clean_text_list[start:len(clean_text_list)])
    toxicity = result["toxicity"]
    toxicity_scores.extend(toxicity)
    start = end
#print(toxicity_scores)
toxicity_scores_series = pd.Series(toxicity_scores, index=len_df)
#print(len(toxicity_scores_series))
# Dem Data Frame zuweisen 
data_frame_topics["Toxicity_score"] = toxicity_scores_series
# Als csv Datei ausgeben
data_frame_topics.to_csv("MAP_twitter_toxicity_topics.csv")

